In [ ]:
import pandas as pd
import numpy as np

from mlxtend.frequent_patterns import fpgrowth, association_rules

df = pd.read_csv('Online_Retail_5000.csv')
df.head()

# gom theo InvoiceNo va StockCode de dem so luong tung mat hang trong moi hoa don
invoice_stockcode = df.groupby(['InvoiceNo', 'StockCode']).size().reset_index(name='Count')
invoice_stockcode.head()


# tính tần số xuất hiện của từng mặt hàng (StockCode) trong các hóa đơn (InvoiceNo)
stockcode_freq = df.groupby('StockCode')['InvoiceNo'].nunique().reset_index(name='Freq')
stockcode_freq.head()

# Tính support cho từng mặt hàng
total_invoices = df['InvoiceNo'].nunique()
stockcode_freq['Support'] = stockcode_freq['Freq'] / total_invoices
stockcode_freq.head()


stockcode_freq_sorted = stockcode_freq.sort_values(by='Support', ascending=False)
stockcode_freq_sorted.head(10)

transactions = []
invoices = df['InvoiceNo'].unique()

for iv in invoices:
    items = df[df['InvoiceNo'] == iv]['StockCode'].unique().tolist()
    transactions.append(items)

print('Total transactions:', len(transactions))


for tr in transactions[:5]:
    print(tr)


from mlxtend.preprocessing import TransactionEncoder
encoder = TransactionEncoder()
onehot = encoder.fit(transactions).transform(transactions)
onehot_df = pd.DataFrame(onehot, columns=encoder.columns_)
onehot_df.head()


# tạo tập các tập mục thường xuyên với ngưỡng support tối thiểu là 0.05
frequent_itemsets = fpgrowth(onehot_df, min_support=0.05, use_colnames=True)
fp_rules = association_rules(frequent_itemsets, metric="support", min_threshold=0.05)
fp_rules_sorted = fp_rules.sort_values(by='support', ascending=False)
print('Number of rules generated:', len(fp_rules_sorted))


print(fp_rules_sorted.head(10))


for i, (index, rule) in enumerate(fp_rules_sorted.head(10).iterrows()):
    X = ', '.join(list(rule['antecedents']))
    Y = ', '.join(list(rule['consequents']))
    support_ = rule['support']
    confidence_ = rule['confidence']
    print(f'Rule {i+1}: {X} -> {Y} (Support: {support_:.4f}, Confidence: {confidence_:.4f})')


